In [ ]:
import os
from pathlib import Path
import warnings

from tqdm import TqdmWarning
warnings.filterwarnings("ignore", category=TqdmWarning)
import pandas as pd
from utils import (
    augment_all_trips,
    build_h3_color_and_position_maps,
    clean_trip_dataset,
    save_h3_maps_to_json,
    save_wave_maps_for_all_trips,
)
os.environ["NCCL_P2P_DISABLE"] = "1"
os.environ["NCCL_IB_DISABLE"] = "1"

PROJECT_ROOT = Path("..")
DATA_DIR = PROJECT_ROOT / "data"
IMAGES_DIR = PROJECT_ROOT / "images"
RAW_TRIPS_CSV = DATA_DIR / "raw_data.csv"
CLEAN_TRIPS_CSV = DATA_DIR / "cleaned_data.csv"
AUGMENTED_TRIPS_CSV = DATA_DIR / "augmented_data.csv"
COLOR_MAP_JSON = DATA_DIR / "h3_color_map.json"
POSITION_MAP_JSON = DATA_DIR / "h3_position_map.json"

# Set to None to keep the default rename behavior from utils.py.
custom_rename_map = None
# Example:
# custom_rename_map = {
#     "LON": "lon",
#     "LAT": "lat",
#     "# Timestamp": "time",
#     "h3_cell_9": "h3",
#     "TRIP": "trip_id",
# }

DATA_DIR.mkdir(parents=True, exist_ok=True)
IMAGES_DIR.mkdir(parents=True, exist_ok=True)

## Preprocessing Dataset

### Loading Trajectory Data

The raw trajectory data is loaded from a CSV file containing vessel movements between ten major ports. Each row represents a temporal observation with columns for timestamp, geographic coordinates (longitude, latitude), the corresponding H3 hexagonal cell identifier, and a unique trip identifier. This dataset forms the foundation for all subsequent preprocessing and model training operations. The DataFrame structure allows efficient filtering by trip and temporal indexing for gap detection and interpolation.

In [ ]:
trips = pd.read_csv(RAW_TRIPS_CSV)
if custom_rename_map:
    preview_rename_map = {
        key: value
        for key, value in custom_rename_map.items()
        if key in trips.columns
    }
    if preview_rename_map:
        trips = trips.rename(columns=preview_rename_map)
trips.head(10)

### Trip Filtering and Column Standardization

Trips are filtered based on duration statistics to remove anomalous trajectories. The column names are standardized to a consistent schema (`time`, `lon`, `lat`, `h3`, `trip_id`) for compatibility with downstream functions. Duration filtering applies a threshold of ±1800 seconds from the mean trip duration, ensuring that extremely short or long voyages—which may represent incomplete data or multi-leg journeys—are excluded from the training set. This yields a clean dataset of representative port-to-port trajectories.

In [ ]:
trips_clean, cleaning_stats = clean_trip_dataset(
    RAW_TRIPS_CSV,
    mean_duration_seconds=13282.0,
    std_duration_seconds=4486.14,
    duration_tolerance_seconds=1800.0,
    output_path=CLEAN_TRIPS_CSV,
    h3_resolution=9,
    rename_map=custom_rename_map,
 )

print(cleaning_stats)
trips_clean.head()

### Temporal Augmentation: Filling Missing Time Bins

Maritime AIS data often contains irregular sampling intervals due to transmission gaps or receiver limitations. The `augment_trip()` function resamples each trajectory onto a regular temporal grid by identifying empty time bins and filling them via linear interpolation from neighboring observations. For each augmented point, the H3 cell is computed using `h3.latlng_to_cell()` at resolution 9, maintaining spatial consistency with the original data. The `augment_all_trips()` wrapper applies this transformation across all trajectories, producing a uniformly sampled dataset suitable for image-based representation.

In [ ]:
trip_augmented_all = augment_all_trips(
    CLEAN_TRIPS_CSV,
    verbose=False,
    n_bins=128,
    h3_resolution=9,
)
trip_augmented_all.to_csv(AUGMENTED_TRIPS_CSV, index=False)

print(f"Saved augmented dataset to {AUGMENTED_TRIPS_CSV}")
print(f"Rows: {len(trip_augmented_all):,}, trips: {trip_augmented_all['trip_id'].nunique():,}")
trip_augmented_all.head()

## Exctract Color Mapping

### Float32 Bit-Packed Colormap for H3 Cells

We encode H3 cell centroids into RGB color space using a deterministic, reversible bit-packing scheme. Geographic coordinates are normalized to [0, 1] based on the bounding box of all observed cells, then quantized to 23 bits per axis (18 high bits + 5 shared bits). The encoding distributes these bits across the RGB channels: the red channel stores the high 18 bits of normalized latitude, the blue channel stores the high 18 bits of normalized longitude, and the green channel stores the combined low 5 bits of both coordinates. An intensity inversion (1.0 − value) followed by scaling to the [0.05, 0.95] range ensures no color component saturates at pure black or white, which are reserved for masking and background.

This design achieves three goals: (1) each H3 cell receives a unique, distinguishable color with no collisions, (2) spatial proximity is approximately preserved in color space—nearby cells have similar hues—and (3) the encoding is fully reversible, allowing downstream models to decode predicted RGB values back to geographic coordinates with sub-meter precision.

In [ ]:
trip_augmented_all = pd.read_csv(AUGMENTED_TRIPS_CSV)

h3_dict, h3_color_map, h3_position_map = build_h3_color_and_position_maps(
    trip_augmented_all,
    lon_col="lon",
    lat_col="lat",
    h3_col="h3",
    plot_graph=True,
)
save_h3_maps_to_json(
    h3_color_map,
    h3_position_map,
    COLOR_MAP_JSON,
    POSITION_MAP_JSON,
)

print(f"Total unique H3 cells: {len(h3_color_map)}")
print(f"Saved H3 color map to {COLOR_MAP_JSON}")
print(f"Saved H3 position map to {POSITION_MAP_JSON}")
list(h3_color_map.items())[:3]

## Save trips as RGB in .npy files (cell must only be executed once)

### Wave Map Generation Function

The `create_wave_map_with_missing()` function converts a single trajectory into a 128×128 float32 image representation. Time is discretized into 128 bins along the vertical axis, with each row representing a fixed temporal window normalized to the longest trip duration in the dataset. Within each row, the 128 columns encode sub-bin temporal positions. Each pixel stores the RGB color corresponding to the vessel's H3 cell at that moment, using the bit-packed colormap defined above. Missing observations or gaps beyond the trip's recorded duration are marked with white pixels (RGB = 1.0, 1.0, 1.0) and tracked in a binary mask array. The function handles coordinate lookups by computing the H3 cell from lon/lat, with a fallback to nearest-neighbor search when a cell falls outside the precomputed colormap vocabulary.

In [ ]:
saved_image_paths, longest_duration = save_wave_maps_for_all_trips(
    trip_augmented_all,
    IMAGES_DIR,
    h3_color_map=h3_color_map,
    h3_position_map=h3_position_map,
    h3_resolution=9,
    n_bins=128,
    save_mask=False,
    verbose=True,
)

print(f"Saved {len(saved_image_paths)} wave maps to {IMAGES_DIR}")
print(f"Longest trip duration: {longest_duration:.2f} seconds")
saved_image_paths[:5]

### Output Summary

This notebook writes the cleaned dataset, the augmented dataset, the wave-map images, and the reusable H3 lookup JSON files required by the training and inference stages.

In [ ]:
print("Artifacts created:")
print(f"Clean dataset: {CLEAN_TRIPS_CSV}")
print(f"Augmented dataset: {AUGMENTED_TRIPS_CSV}")
print(f"Images directory: {IMAGES_DIR}")
print(f"H3 color map JSON: {COLOR_MAP_JSON}")
print(f"H3 position map JSON: {POSITION_MAP_JSON}")
print(f"Wave maps saved: {len(saved_image_paths)}")